# GPPO Phase-1 完整正式复现（展开版）

本 Notebook 按阶段展示：环境配置、断点恢复、预检、正式矩阵、实时进度、Drive 备份、test100、通信重放、曲线、报告、审计与打包。请选择 **T4 GPU 或更高**，然后按顺序运行。

正式范围：四尺度 × 五训练种子 × PPO/GPPO/NoGate/SingleHead/Adaptive-score；每个可学习模型 2000 iterations。


## 1. 挂载 Drive 与全局配置


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, time
from datetime import timedelta
from pathlib import Path
from IPython.display import HTML, Image, Markdown, display
import pandas as pd

REPO_URL = 'https://github.com/Battleplus/GPPO.git'
BRANCH = '8.8-GPPO无偏好'
REPO = Path('/content/GPPO')
LOCAL_ROOT = Path('/content/phase1_formal')
DRIVE_BASE = Path('/content/drive/MyDrive')
DRIVE_ROOT = DRIVE_BASE / 'GPPO_phase1_formal'
MIGRATION_ZIP = DRIVE_BASE / 'GPPO_phase1_migration.zip'
JOBS = 2
DEVICE = 'cuda'
BACKUP_SECONDS = 600
PROGRESS_SECONDS = 20
EXPECTED_RUNS = 135
ITERATIONS_PER_RUN = 2000
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive:', DRIVE_ROOT)
print('Migration:', MIGRATION_ZIP, MIGRATION_ZIP.exists())


## 2. 获取固定版本代码、安装依赖并验证 GPU


In [ ]:
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt'),'pandas','matplotlib','scipy','tqdm'], check=True)
sys.path.insert(0, str(REPO/'colab'))
from run_gppo_phase1_full_colab import (
    run_checked, restore_state, discover_progress, gpu_status, show_progress, postprocess
)
import torch
assert torch.cuda.is_available(), '请切换到 T4 GPU 或更高运行时'
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('Commit:', COMMIT)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('GPU status:', gpu_status())


## 3. 恢复迁移包与本地断点，并验证数量


In [ ]:
restore_state(LOCAL_ROOT, DRIVE_ROOT, MIGRATION_ZIP)
marker = LOCAL_ROOT/'T5-10-48_literal_event'/'PHASE1_CANDIDATE_RESELECTION.json'
frozen = list((LOCAL_ROOT/'T5-10-48_literal_event').rglob('checkpoint_phase1_frozen.pt'))
resumes = list((LOCAL_ROOT/'T5_primary').rglob('resume_latest.pt'))
print('协议文件存在:', marker.exists())
print('冻结 GPPO checkpoints:', len(frozen))
print('PPO 续训 checkpoints:', len(resumes))
for path in frozen + resumes:
    print(' -', path.relative_to(LOCAL_ROOT), f'{path.stat().st_size/1024/1024:.2f} MiB')
shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
print('✅ 迁移状态已验证并备份')


## 4. 公式、环境、CUDA 与断点恢复预检


In [ ]:
PREFLIGHT_LOG = DRIVE_ROOT/'preflight_tests.log'
run_checked([
    sys.executable, '-m', 'pytest',
    'tests/test_paper_faithful.py',
    'tests/test_paper_faithful_baselines.py',
    'tests/test_paper_faithful_resume.py',
    'tests/test_paper_faithful_cuda.py', '-q'
], REPO, PREFLIGHT_LOG)
print(PREFLIGHT_LOG.read_text(encoding='utf-8', errors='replace')[-5000:])
print('✅ 预检通过，允许启动正式训练')


## 5. 启动完整正式矩阵

该调度器会自动依次完成 T5 主矩阵、其余三个尺度、Adaptive-score、test100 和四种通信模式重放。


In [ ]:
MATRIX_LOG = DRIVE_ROOT/'colab_phase1_matrix.log'
MATRIX_COMMAND = [
    sys.executable, str(REPO/'run_phase1_formal_matrix.py'),
    '--frozen-protocol', str(REPO/'configs/PHASE1_FROZEN_PROTOCOL.json'),
    '--formal-root', str(LOCAL_ROOT),
    '--legacy-literal-root', str(LOCAL_ROOT/'T5-10-48_literal_event'),
    '--python', sys.executable, '--jobs', str(JOBS), '--device', DEVICE
]
print(' '.join(MATRIX_COMMAND))
matrix_stream = MATRIX_LOG.open('a', encoding='utf-8')
matrix_process = subprocess.Popen(
    MATRIX_COMMAND, cwd=REPO, stdout=matrix_stream, stderr=subprocess.STDOUT, text=True
)
print('✅ 调度器已启动，PID:', matrix_process.pid)
print('日志:', MATRIX_LOG)


## 6. 实时训练进度、Reward、Makespan、GPU、速度、ETA 与 Drive 备份


In [ ]:
started = time.monotonic()
last_backup = started
previous_total = previous_time = smoothed_rate = None
status_display = display(HTML('<h3>正在读取训练进度……</h3>'), display_id=True)
table_display = display(pd.DataFrame(), display_id=True)
try:
    while matrix_process.poll() is None:
        now = time.monotonic()
        rows = discover_progress(LOCAL_ROOT)
        total = sum(
            min(ITERATIONS_PER_RUN, int(row['iteration']))
            for row in rows if 'T5-10-48_literal_event' not in str(row['run'])
        )
        if previous_total is not None:
            rate = max(0, total-previous_total) / max(1.0, now-previous_time)
            if rate > 0:
                smoothed_rate = rate if smoothed_rate is None else 0.8*smoothed_rate + 0.2*rate
        previous_total, previous_time = total, now
        remaining = max(0, EXPECTED_RUNS*ITERATIONS_PER_RUN-total)
        eta = str(timedelta(seconds=int(remaining/smoothed_rate))) if smoothed_rate else '正在估算'
        final_count = len(list(LOCAL_ROOT.rglob('checkpoint.pt')))
        resume_count = len(list(LOCAL_ROOT.rglob('resume_latest.pt')))
        test_count = len(list(LOCAL_ROOT.rglob('test_native_100.json')))
        status_display.update(HTML(
            f'<h3>GPPO Phase-1 正式流水线运行中</h3>'
            f'<p><b>总进度：</b>{total:,}/{EXPECTED_RUNS*ITERATIONS_PER_RUN:,} '
            f'({100*total/(EXPECTED_RUNS*ITERATIONS_PER_RUN):.2f}%)</p>'
            f'<p><b>训练目录：</b>{len(rows)} | <b>最终 checkpoint：</b>{final_count} | '
            f'<b>续训 checkpoint：</b>{resume_count} | <b>test100：</b>{test_count}</p>'
            f'<p><b>已用：</b>{timedelta(seconds=int(now-started))} | '
            f'<b>速度：</b>{(smoothed_rate or 0):.3f} iter/s | <b>ETA：</b>{eta}</p>'
            f'<p><b>GPU：</b>{gpu_status()}</p>'
        ))
        if rows:
            frame = pd.DataFrame(rows).sort_values(['updated','iteration'], ascending=[False,False]).head(30)
            table_display.update(frame)
        if now-last_backup >= BACKUP_SECONDS:
            shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
            last_backup = now
            print(time.strftime('%H:%M:%S'), '✅ 已备份可恢复状态到 Drive')
        time.sleep(PROGRESS_SECONDS)
finally:
    matrix_stream.close()
    shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
return_code = matrix_process.wait()
if return_code != 0:
    print('\n'.join(MATRIX_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-150:]))
    raise RuntimeError('正式矩阵失败；修复后重新运行第 3-6 阶段即可自动续训')
status_display.update(HTML('<h2 style="color:green">✅ 训练、test100 与通信重放全部完成</h2>'))


## 7. 生成汇总、三张正式曲线、审计与中英文报告


In [ ]:
postprocess(REPO, LOCAL_ROOT)
outputs = [
    'formal_summary.json', 'formal_artifact_audit.json',
    'formal_report.md', 'GPPO_REPRODUCTION_REPORT_ZH.md',
    'fig8_reward.png', 'fig8_realized_makespan.png',
    'fig8_validation_makespan.png'
]
for name in outputs:
    path = LOCAL_ROOT/name
    print(('✅' if path.exists() else '❌'), name)
audit = json.loads((LOCAL_ROOT/'formal_artifact_audit.json').read_text(encoding='utf-8'))
print('产物审计 valid:', audit.get('valid'))


## 8. 展示 Reward/Makespan 曲线与中文报告


In [ ]:
for name in ['fig8_reward.png','fig8_realized_makespan.png','fig8_validation_makespan.png']:
    path = LOCAL_ROOT/name
    if path.exists():
        display(Markdown(f'### {name}'))
        display(Image(filename=str(path)))
report_path = LOCAL_ROOT/'GPPO_REPRODUCTION_REPORT_ZH.md'
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8', errors='replace')[:30000]))


## 9. 最终 Drive 备份与 ZIP 打包


In [ ]:
shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
archive = shutil.make_archive(
    str(DRIVE_BASE/'GPPO_phase1_formal_complete'), 'zip', root_dir=LOCAL_ROOT
)
print('✅ Drive 结果目录:', DRIVE_ROOT)
print('✅ 完整压缩包:', archive)
print('✅ Reward 曲线:', DRIVE_ROOT/'fig8_reward.png')
print('✅ 中文报告:', DRIVE_ROOT/'GPPO_REPRODUCTION_REPORT_ZH.md')
print('✅ 审计结果:', DRIVE_ROOT/'formal_artifact_audit.json')


## 阶段边界

该 Notebook 完整覆盖 **Phase 1：GPPO 基线复现与验收**。Phase 2 PCRL 需要新增向量奖励、偏好条件策略和多目标优化；Phase 3 世界模型需要事件数据集、预测模型和触发校准。只有 Phase 1 验收通过后才进入后两阶段，不能把它们伪装成几个参数直接接在本训练后面。
